
# Bias Analysis — Credit Approval Decisions

This notebook reviews past credit approval outcomes to check for gender-related differences.  
We focus on two things: (1) **disparate impact (DI)** in observed approvals, and (2) whether some non-sensitive features may still act as **proxies** for gender.



## 1. Data & Setup

We run the analysis on the **analysis-ready dataset** produced by the data-quality pipeline.  
First, we check the dataset size and column names so we know exactly what we can test.

**Note:** The dataset does not include age (or date of birth). For that reason, the fairness checks in this notebook focus on **gender**.

In [17]:
import sys
from pathlib import Path
import importlib
import pandas as pd
from scipy.stats import chi2_contingency

sys.path.append(str(Path("..").resolve()))

import src.data_clean_notebook as preproc
importlib.reload(preproc)

df = preproc.run_data_quality_pipeline("../data/raw_credit_applications.json")

Final dataset shape: (502, 27)


In [18]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())

shape: (502, 27)
columns: ['applicant_gender', 'applicant_zip_code', 'fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'decision_loan_approved', 'spend_shopping', 'spend_rent', 'spend_alcohol', 'spend_txn_count', 'spend_total', 'spend_mean', 'spend_max', 'spend_unique_cats', 'spend_dining', 'spend_healthcare', 'spend_fitness', 'spend_entertainment', 'spend_insurance', 'spend_travel', 'spend_transportation', 'spend_utilities', 'spend_groceries', 'spend_education', 'spend_adult_entertainment', 'spend_gambling']


In [19]:
# Minimal sanity checks
print("\nRaw gender counts:")
print(df["applicant_gender"].value_counts(dropna=False))

print("\nRaw label counts:")
print(df["decision_loan_approved"].value_counts(dropna=False))


Raw gender counts:
applicant_gender
Male      195
Female    193
F          58
M          53
NaN         3
Name: count, dtype: int64

Raw label counts:
decision_loan_approved
True     292
False    210
Name: count, dtype: int64


## 2. Definitions

- **Outcome (y):** loan approval label (1 = approved, 0 = rejected), based on **decision_loan_approved**.
- **Protected attribute:** gender from **applicant_gender**, normalized into **gender_norm** (male/female).
- **Selection rate:** approval rate within each group (used for DI).

We compute fairness metrics only when **both** the protected attribute and the outcome are available.  
Rows with missing/unmapped gender are excluded from DI because DI is undefined without group membership.

In [39]:
# Create binary outcome label y (1=approved, 0=rejected)

df["y"] = df["decision_loan_approved"].astype(int)

## 3. Missingness & Validity Checks

Before computing fairness metrics, we check missingness in the protected attribute.  
If missing gender is linked to approval outcomes, dropping those rows could bias the DI estimate.

We report:
- missing rate for **applicant_gender**
- approval rate for **missing vs non-missing** gender rows

In [21]:
# Missingness audit on raw applicant_gender

raw_gender = df["applicant_gender"]
miss_raw_gender = raw_gender.isna() | (raw_gender.astype(str).str.strip() == "")

print(f"Missing applicant_gender rate: {miss_raw_gender.mean():.4%}")
print("Counts (missing vs non-missing):")
print(miss_raw_gender.value_counts())


print("\nApproval rate when raw gender is missing:", df.loc[miss_raw_gender, "y"].mean() if miss_raw_gender.any() else "N/A")
print("Approval rate when raw gender is NOT missing:", df.loc[~miss_raw_gender, "y"].mean())

Missing applicant_gender rate: 0.5976%
Counts (missing vs non-missing):
applicant_gender
False    499
True       3
Name: count, dtype: int64

Approval rate when raw gender is missing: 0.6666666666666666
Approval rate when raw gender is NOT missing: 0.5811623246492986


**Missingness audit**

The missing rate for "applicant_gender" is **0.60%**, which is very small.  
Approval is **0.667** for the missing-gender rows versus **0.581** for non-missing rows.  
Since the missing group has only 3 records, excluding these rows is unlikely to change DI in a meaningful way — but we report it for completeness.

## 4. Gender Snapshot (before DI)

Before computing DI, we summarise the basic group picture:
- how many applicants are in each gender group
- the approval rate in each group

These numbers make the DI result easier to interpret.

In [43]:
# Create gender_norm WITHOUT hiding missing values

g = df["applicant_gender"].astype("string").str.strip().str.lower()

gender_map = {
    "m": "male", "male": "male", "man": "male",
    "f": "female", "female": "female", "woman": "female",
}

df["gender_norm"] = g.map(gender_map)

# df["gender_norm"] remains NA if raw gender was missing or unrecognized

#  show what values are not mapped
print("gender_norm value counts (including NA):")
print(df["gender_norm"].value_counts(dropna=False))

gender_norm value counts (including NA):
gender_norm
female    251
male      248
NaN         3
Name: count, dtype: int64


## 5. Disparate Impact (DI) by Gender

We compute selection rates by gender and then report DI using the common four-fifths rule as a screening threshold.

### 5.1 Method

We compute **Disparate Impact (DI)** as:

$$
DI = \frac{\text{selection rate of the unprivileged group}}{\text{selection rate of the privileged group}}
$$

Convention used here:
- **privileged** = male
- **unprivileged** = female

DI is computed only when both "gender_norm" and "y" are present.

### 5.2 Computation (selection rates + DI)

We first compute group sizes and approval rates, then calculate DI (female/male) and check the four-fifths rule.

In [45]:
# Prepare DI dataset 

tmp = df[["gender_norm", "y"]].dropna().copy()
tmp["y"] = tmp["y"].astype(int)

print("DI dataset size:", tmp.shape[0])
print(tmp["gender_norm"].value_counts())

group_sizes = tmp["gender_norm"].value_counts()
approval_rates = tmp.groupby("gender_norm")["y"].mean().sort_values(ascending=False)

print("Group sizes:\n", group_sizes)
print("\nApproval rates:\n", approval_rates)

DI dataset size: 499
gender_norm
female    251
male      248
Name: count, dtype: int64
Group sizes:
 gender_norm
female    251
male      248
Name: count, dtype: int64

Approval rates:
 gender_norm
male      0.657258
female    0.505976
Name: y, dtype: float64


In [24]:
priv_group = "male"
unpriv_group = "female"

if priv_group in approval_rates.index and unpriv_group in approval_rates.index:
    di = float(approval_rates.loc[unpriv_group] / approval_rates.loc[priv_group])
    print(f"DI (female/male) = {di:.4f}")
    print(f"80% rule flag (DI < 0.8): {di < 0.8}")
else:
    print("Cannot compute DI: missing male or female group in the DI dataset.")
    print("Available groups:", approval_rates.index.tolist())

DI (female/male) = 0.7698
80% rule flag (DI < 0.8): True


### 5.3 Interpreting DI 

As a simple screening rule:
- **DI < 0.80** flags potential adverse impact.

DI is descriptive; it highlights differences in observed outcomes, but it does not explain why they happen.  
That’s why we follow up with association tests and proxy checks.

### 5.4 Conclusion 

- On the DI computation subset, the approvalrate is **0.657** for male (n=248) and **0.506** for female (n=251).

- Using the common convention DI = female_rate / male_rate, we obtain **DI = 0.7698<0.8** indicating potential adverse impact against the female group in the observed decision outcomes.

- DI is a screening metric, not a causal proof of discrimination. So next we will investigate possible proxy drivers and checking whether the disparity persists after controlling for relevant risk factors.

## 6. Statistical Association (Gender vs Approval)

DI shows a gap in approval rates. Here we check whether that gap is also supported statistically.

We report:
- a **chi-square test** on the gender × approval contingency table
- **Cramér’s V** as an effect size (how strong the association is)

**H0:** gender and approval outcomes are independent.  
If **p < 0.05**, we reject H0.

### 6.1 Chi-square test of independence

In [25]:


ct = pd.crosstab(tmp["gender_norm"], tmp["y"])
print("Contingency table (gender x approval):\n", ct)

chi2, p, dof, expected = chi2_contingency(ct)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
print("\nExpected counts under H0 (independence):\n", expected_df.round(2))

Contingency table (gender x approval):
 y              0    1
gender_norm          
female       124  127
male          85  163

Chi-square statistic: 11.1156
p-value: 0.000856
Degrees of freedom: 1

Expected counts under H0 (independence):
 y                 0       1
gender_norm                
female       105.13  145.87
male         103.87  144.13


**Chi-square conclusion**

The chi-square test suggests approval outcomes are not independent of gender (**χ²(1)=11.12, p=0.000856**).  
This aligns with the DI result: the observed approval rate is higher for males than for females.  
The test supports treating the gender gap as more than random noise in this sample.

### 6.2 Cramer's V test
We report Cramer’s V to quantify the strength of association between gender and approval.

In [26]:
import numpy as np

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))
print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.1493


**Cramér’s V conclusion**

Cramér’s V is **0.1493**, which indicates a small-to-moderate association between gender and approval.  
Even a modest association can matter in a decision setting — especially when DI also breaches the four-fifths rule — so it’s worth monitoring and reviewing.

## 7. Robustness Checks

We run a few quick checks to make sure the DI result is not driven by simple choices such as:
- the exact gender-normalization mapping
- dropping the small number of missing gender rows
- whether any unknown/non-binary values are included or excluded

In [46]:
# DI computed on non-missing gender
baseline_n = tmp.shape[0]
print("Baseline DI dataset size:", baseline_n)

# few missing gender rows, report it explicitly
missing_gender_n = df["gender_norm"].isna().sum()
print("Rows with missing/unmapped gender_norm:", int(missing_gender_n))

Baseline DI dataset size: 499
Rows with missing/unmapped gender_norm: 3


**Robutness Conclusion**

Only **3/502** rows have missing/unmapped gender, so the DI estimate is not driven by dropping a large portion of the data.

## 8. Proxy Discrimination Analysis

Even if gender is not used directly, other features can still pick up gender-related patterns.  
We use a simple two-link check:

1) **Feature ↔ Gender:** does the feature vary meaningfully across gender groups?  
2) **Feature ↔ Outcome:** does the feature also relate to approval outcomes?

A proxy risk is more convincing when **both** links show evidence.

### 8.1 Sanity check

Before proxy tests, we confirm **y**, **gender_norm**, and **applicant_zip_code** are present and look sensible (including missingness).

In [49]:
# required variables exist and look reasonable

required = ["y", "gender_norm", "applicant_zip_code"]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for proxy analysis: {missing_cols}")

print("y value counts:")
print(df["y"].value_counts(dropna=False))

print("\ngender_norm value counts (incl. NA):")
print(df["gender_norm"].value_counts(dropna=False))

print("\nZIP code missing rate:", df["applicant_zip_code"].isna().mean())

y value counts:
y
1    292
0    210
Name: count, dtype: int64

gender_norm value counts (incl. NA):
gender_norm
female    251
male      248
NaN         3
Name: count, dtype: int64

ZIP code missing rate: 0.00398406374501992



We verify that the proxy analysis uses consistent definitions of **y** (approval outcome) and **gender_norm** (protected attribute), and that **applicant_zip_code** is available with low missingness.

### 8.2 ZIP code ↔ Gender

We check whether ZIP codes differ by gender.  
If they do, ZIP may encode demographic information and can act as a proxy.

In [32]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

zip_gender = df[["applicant_zip_code", "gender_norm"]].dropna().copy()
zip_gender["applicant_zip_code"] = zip_gender["applicant_zip_code"].astype(str)

ct_zg = pd.crosstab(zip_gender["applicant_zip_code"], zip_gender["gender_norm"])
print("Contingency table shape (ZIP x gender):", ct_zg.shape)

chi2_zg, p_zg, dof_zg, exp_zg = chi2_contingency(ct_zg)

n = ct_zg.to_numpy().sum()
r, k = ct_zg.shape
cramers_v_zg = np.sqrt(chi2_zg / (n * (min(r - 1, k - 1))))

print(f"Chi-square (ZIP vs gender): {chi2_zg:.4f}")
print(f"p-value: {p_zg:.6f}")
print(f"Cramer's V: {cramers_v_zg:.4f}")

# Sparsity diagnostics (χ² reliability indicator)
print(f"Min expected count: {exp_zg.min():.2f}")
print(f"Share of expected cells < 5: {(exp_zg < 5).mean():.2%}")

Contingency table shape (ZIP x gender): (195, 2)
Chi-square (ZIP vs gender): 393.7343
p-value: 0.000000
Cramer's V: 0.8883
Min expected count: 0.50
Share of expected cells < 5: 100.00%


**ZIP ↔ gender conclusion**

ZIP and gender look associated (p < 0.001).  
However, the contingency table is extremely sparse (min expected = **0.50**, and **100%** of expected cells are < 5).  
Because the chi-square assumptions are badly violated here, the p-value and Cramér’s V are **suggestive**, not definitive.

Still, the pattern is enough to keep ZIP on the list of proxy candidates for follow-up checks.

### 8.3 ZIP code ↔ Approval outcome

ZIP has many categories, so we group ZIPs (top 20 + OTHER) to reduce sparsity.  
We then compare approval rates across groups and test the grouped association.

In [48]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# Build ZIP stats
zip_out = df[["applicant_zip_code", "y"]].dropna().copy()
zip_out["applicant_zip_code"] = zip_out["applicant_zip_code"].astype(str)
zip_out["y"] = zip_out["y"].astype(int)

zip_counts = zip_out["applicant_zip_code"].value_counts()
top_k = 20  
top_zips = set(zip_counts.head(top_k).index)

zip_out["zip_group"] = np.where(zip_out["applicant_zip_code"].isin(top_zips),zip_out["applicant_zip_code"],"OTHER")

# Approval rates by grouped ZIP
group_stats = zip_out.groupby("zip_group")["y"].agg(n="count", approval_rate="mean").sort_values("n", ascending=False)
display(group_stats.head(25))

# Chi-square on grouped table (less sparse)
ct = pd.crosstab(zip_out["zip_group"], zip_out["y"])
chi2, p, dof, exp = chi2_contingency(ct)

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

print("Grouped contingency table shape (zip_group x y):", ct.shape)
print(f"Chi-square (grouped ZIP vs approval): {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Cramer's V: {cramers_v:.4f}")
print(f"Min expected count: {exp.min():.2f}")
print(f"Share of expected cells < 5: {(exp < 5).mean():.2%}")

,n,approval_rate
zip_group,,
OTHER,389,0.570694
10048,8,0.750000
90284,7,0.428571
10096,7,0.285714
10004,6,1.000000
10057,6,0.500000
10020,6,0.666667
10019,6,0.333333
10002,5,0.600000


Grouped contingency table shape (zip_group x y): (21, 2)
Chi-square (grouped ZIP vs approval): 25.9893
p-value: 0.166167
Cramer's V: 0.2280
Min expected count: 2.09
Share of expected cells < 5: 95.24%


**ZIP ↔ approval conclusion (grouped ZIPs)**

After grouping ZIPs (top 20 + OTHER), the ZIP group–approval association is **not statistically significant** (p = **0.166**).  
Cramér’s V is **0.228**, but the grouped table is still sparse (min expected = **2.09**, and **95.24%** of expected cells are < 5).  

Overall, we do not see strong evidence that ZIP is a direct driver of approval outcomes in this dataset.

### 8.5 Annual income (financial candidate)

ZIP-based tests are limited by sparsity, so we also look at a financial feature that is more directly tied to credit decisions.  
We apply the same two-link check: income ↔ gender, and income ↔ approval.

In [50]:
# Income and Gender (group differences)

income_gender = df[["gender_norm", "fin_annual_income"]].dropna().copy()
income_gender = income_gender[income_gender["gender_norm"].isin(["male", "female"])]

summary_income = income_gender.groupby("gender_norm")["fin_annual_income"].agg(
    n="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
)

display(summary_income)

# Simple standardized mean difference (SMD) as effect size
male_income = income_gender.loc[income_gender["gender_norm"] == "male", "fin_annual_income"]
female_income = income_gender.loc[income_gender["gender_norm"] == "female", "fin_annual_income"]

pooled_std = np.sqrt((male_income.var(ddof=1) + female_income.var(ddof=1)) / 2)
smd = (female_income.mean() - male_income.mean()) / pooled_std if pooled_std > 0 else np.nan

print(f"SMD (female - male) for income: {smd:.3f}")

,n,mean,median,std,min,max
gender_norm,,,,,,
female,246,84253.791826,83000.0,28304.842591,22000.0,171000.0
male,247,81388.663968,81000.0,27486.632986,22000.0,168000.0


SMD (female - male) for income: 0.103


In [51]:
# approval rate by income quantiles

income_out = df[["y", "fin_annual_income"]].dropna().copy()
income_out["y"] = income_out["y"].astype(int)

# 5 quantile bins 
income_out["income_bin"] = pd.qcut(income_out["fin_annual_income"], q=5, duplicates="drop")

bin_stats = income_out.groupby("income_bin")["y"].agg(n="count", approval_rate="mean")
display(bin_stats)

# overall correlation check 
corr = income_out[["fin_annual_income", "y"]].corr().iloc[0, 1]
print(f"Correlation(income, approval y): {corr:.3f}")

/var/folders/6m/9yqj6yd90d10n4czxh4lntpc0000gn/T/ipykernel_12409/3368507641.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_stats = income_out.groupby("income_bin")["y"].agg(n="count", approval_rate="mean")


,n,approval_rate
income_bin,,
"(21999.999, 58000.0]",101,0.396040
"(58000.0, 75000.0]",101,0.544554
"(75000.0, 89000.0]",99,0.666667
"(89000.0, 105000.0]",97,0.649485
"(105000.0, 171000.0]",98,0.663265


Correlation(income, approval y): 0.178


**Income conclusion**

**Link (1): Income ↔ Gender.** Male and female income levels are very similar (SMD = **0.103**), so income shows only a small gender difference in this sample.

**Link (2): Income ↔ Approval.** Approval increases noticeably across income bins (from **0.396** in the lowest bin to ~**0.66** in higher bins), and the correlation is positive (**r = 0.178**).

**Overall:** Income looks like a legitimate, risk-related driver of approval, rather than a strong gender proxy here.

### 8.6 Debt-to-income ratio (DTI)

We repeat the two-link proxy check for DTI: DTI ↔ gender and DTI ↔ approval.

In [37]:
# 8.6.1 DTI ↔ Gender (group differences)

dti_gender = df[["gender_norm", "fin_debt_to_income"]].dropna().copy()
dti_gender = dti_gender[dti_gender["gender_norm"].isin(["male", "female"])]

summary_dti = dti_gender.groupby("gender_norm")["fin_debt_to_income"].agg(
    n="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
)
display(summary_dti)

male_dti = dti_gender.loc[dti_gender["gender_norm"] == "male", "fin_debt_to_income"]
female_dti = dti_gender.loc[dti_gender["gender_norm"] == "female", "fin_debt_to_income"]

pooled_std = np.sqrt((male_dti.var(ddof=1) + female_dti.var(ddof=1)) / 2)
smd_dti = (female_dti.mean() - male_dti.mean()) / pooled_std if pooled_std > 0 else np.nan
print(f"SMD (female - male) for DTI: {smd_dti:.3f}")

,n,mean,median,std,min,max
gender_norm,,,,,,
female,251,0.243865,0.22,0.154939,0.05,1.85
male,248,0.248952,0.24,0.114381,0.05,0.45


SMD (female - male) for DTI: -0.037


In [53]:
# approval rate by DTI quantiles

dti_out = df[["y", "fin_debt_to_income"]].dropna().copy()
dti_out["y"] = dti_out["y"].astype(int)

dti_out["dti_bin"] = pd.qcut(dti_out["fin_debt_to_income"], q=5, duplicates="drop")
dti_bin_stats = dti_out.groupby("dti_bin", observed=True)["y"].agg(n="count", approval_rate="mean")
display(dti_bin_stats)

corr_dti = dti_out[["fin_debt_to_income", "y"]].corr().iloc[0, 1]
print(f"Correlation(DTI, approval y): {corr_dti:.3f}")

,n,approval_rate
dti_bin,,
"(0.049, 0.12]",103,0.631068
"(0.12, 0.2]",111,0.558559
"(0.2, 0.28]",95,0.526316
"(0.28, 0.37]",102,0.607843
"(0.37, 1.85]",91,0.582418


Correlation(DTI, approval y): 0.007


**DTI conclusion**

**Link (1): DTI ↔ Gender.** DTI is almost the same across gender groups (SMD = **-0.037**), so it carries little gender signal.

**Link (2): DTI ↔ Approval.** Approval rates across DTI bins do not show a clear pattern, and the correlation is essentially zero (**r = 0.007**).

**Overall:** DTI is neither a strong gender proxy nor a strong driver of approval in this dataset.

## 8.7 Proxy risk summary (ZIP + financial variables)

Using the two-link logic (feature ↔ gender, feature ↔ outcome):

- **ZIP:** shows signs of gender-encoding, but inference is limited by severe sparsity. After grouping, we still do not find strong evidence that ZIP relates to approval (p = 0.166).
- **Income:** shows only a small gender difference (SMD ≈ 0.10), while being clearly related to approval (corr ≈ 0.178). This looks more like a risk-related driver than a proxy.
- **DTI:** shows negligible gender difference (SMD ≈ -0.04) and near-zero outcome association (corr ≈ 0.007).

**Takeaway:** Among the candidates we tested, income is the clearest outcome-relevant feature. Evidence for a proxy pathway via ZIP or DTI is limited under these tests, so the focus should remain on monitoring outcome gaps (DI) and being cautious with features that can encode demographics (e.g., geographic granularity).

## 9. Limitations

- **This is not causal evidence.** DI and the tests here describe patterns in the data. They can flag a potential problem, but they do not tell us *why* it happens or whether there was intent.
- **No age-based DI.** The analysis-ready dataset does not include age (or date of birth), so we cannot run the same checks for age groups.
- **ZIP is hard to test cleanly.** ZIP has many categories, which makes the contingency tables sparse. Grouping helps, but the statistical conclusions are still fragile.
- **Limited sample.** Results are based on n=502 records and may reflect noise or artefacts from data simulation.

## 10. Recommendations

- **Monitor DI over time.** Track approval rates by gender and keep an eye on DI. When **DI < 0.8** (four-fifths rule), trigger a review.
- **Dig into what drives the gap.** Income is clearly related to approval, so check whether the gender difference remains **within similar income bands** (stratified DI) and/or after controlling for a small set of key risk variables.
- **Treat geographic granularity with care.** If ZIP is used, justify why it is needed. Consider reducing it to a coarser level (e.g., region) to lower proxy risk.
- **Add basic process controls.** For borderline cases, keep audit logs and introduce human review so decisions are explainable and accountable.

## 11. Key Findings Summary for README 

- Gender is missing in **3/502 (0.60%)** rows.
- On the non-missing gender subset (n=499), approval rates are **0.657 for male (n=248)** and **0.506 for female (n=251)**.
- **DI (female/male) = 0.7698**, which falls below **0.80** and triggers the four-fifths rule.

**Proxy checks (two-link logic):**
- **ZIP:** looks related to gender, but the ZIP×gender table is extremely sparse. After grouping ZIPs (top 20 + OTHER), we do **not** see strong evidence that ZIP is tied to approval (**p = 0.166**, **Cramér’s V = 0.228**), so a clear proxy outcome pathway is not confirmed.
- **Income:** shows only a small gender difference (**SMD = 0.103**) but is clearly linked to approval (approval rises from **0.396** in the lowest income bin to ~**0.66** in higher bins; **corr = 0.178**). This looks more like a legitimate driver than a gender proxy.
- **DTI:** shows negligible gender difference (**SMD = -0.037**) and almost no link to approval (**corr = 0.007**).